# 03 策略回测与分析
以双均线策略为例，展示完整回测流程，并深入分析交易记录与绩效指标。

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from futuresquant.data.loader import FuturesDataLoader
from futuresquant.backtest.engine import BacktestEngine, BacktestConfig
from futuresquant.backtest.analyzer import PerformanceAnalyzer
from futuresquant.strategy.examples.ma_cross import MACrossStrategy

DATA_DIR = r'I:\stock\FuturesQuant\raw_data\1min_FU'
SYMBOL = 'FU2210'

loader = FuturesDataLoader(DATA_DIR)
klines = loader.load(SYMBOL, start='2022-01-01', end='2022-10-31')
print(f'K线行数: {len(klines):,}')

## 1. 运行回测

In [ ]:
strategy = MACrossStrategy(fast=5, slow=20)
config = BacktestConfig(
    symbol=SYMBOL,
    initial_capital=1_000_000,
    slippage_ticks=1,
)

engine = BacktestEngine(strategy, config)
result = engine.run(klines)

# 打印绩效摘要
result.print_summary()

## 2. 权益曲线 & 回撤

In [ ]:
equity = result.account.equity_curve()
analyzer = PerformanceAnalyzer(equity, config.initial_capital)
drawdown = analyzer.drawdown_series()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['权益曲线', '回撤 (%)', '标的价格'],
                    row_heights=[0.5, 0.25, 0.25], vertical_spacing=0.06)

fig.add_trace(go.Scatter(x=equity.index, y=equity.values,
                          name='权益', line=dict(color='#1f77b4')), row=1, col=1)
fig.add_hline(y=config.initial_capital, line_dash='dash',
               line_color='gray', row=1, col=1)

fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown.values * 100,
                          name='回撤', fill='tozeroy',
                          line=dict(color='#d62728'),
                          fillcolor='rgba(214,39,40,0.2)'), row=2, col=1)

fig.add_trace(go.Scatter(x=klines.index, y=klines['close'],
                          name='收盘价', line=dict(color='#2ca02c', width=1)), row=3, col=1)

fig.update_layout(title=f'双均线策略回测 — {SYMBOL}', height=700, hovermode='x unified')
fig.update_yaxes(title_text='元', row=1, col=1)
fig.update_yaxes(title_text='%', row=2, col=1)
fig.show()

## 3. 逐笔成交分析

In [ ]:
from futuresquant.backtest.order import Direction, Offset

fills = result.account.fills
fills_df = pd.DataFrame([
    {
        'time': f.time,
        'direction': f.direction.value,
        'offset': f.offset.value,
        'price': f.price,
        'volume': f.volume,
        'commission': f.commission,
    }
    for f in fills
])

print(f'总成交笔数: {len(fills_df)}')
print(f'总手续费: {fills_df["commission"].sum():.2f} 元')
fills_df.head(10)

In [ ]:
# 在价格图上标注开仓/平仓点
opens = fills_df[fills_df['offset'] == 'OPEN']
closes = fills_df[fills_df['offset'] == 'CLOSE']

buy_opens = opens[opens['direction'] == 'LONG']
sell_opens = opens[opens['direction'] == 'SHORT']

# 采样绘图（数据量大时只显示前 500 笔）
price_sample = klines['close'].resample('5min').last().dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(x=price_sample.index, y=price_sample.values,
                          name='价格(5min)', line=dict(color='gray', width=1)))

fig.add_trace(go.Scatter(
    x=buy_opens['time'].iloc[:200], y=buy_opens['price'].iloc[:200],
    mode='markers', name='买入开仓',
    marker=dict(symbol='triangle-up', size=8, color='#2ca02c')
))
fig.add_trace(go.Scatter(
    x=sell_opens['time'].iloc[:200], y=sell_opens['price'].iloc[:200],
    mode='markers', name='卖出开仓',
    marker=dict(symbol='triangle-down', size=8, color='#d62728')
))

fig.update_layout(title='成交点标注（前200笔开仓）', height=450)
fig.show()

## 4. 参数敏感性分析
遍历 fast/slow 参数组合，找出 Sharpe 最优区域。

In [ ]:
fast_range = [3, 5, 10, 15]
slow_range = [20, 30, 40, 60]

records = []
for fast in fast_range:
    for slow in slow_range:
        if fast >= slow:
            continue
        res = BacktestEngine(
            MACrossStrategy(fast, slow),
            BacktestConfig(SYMBOL, initial_capital=1_000_000)
        ).run(klines)
        m = res.metrics
        records.append({
            'fast': fast, 'slow': slow,
            'sharpe': m['sharpe'],
            'total_return': m['total_return'],
            'max_drawdown': m['max_drawdown'],
        })

param_df = pd.DataFrame(records)
print(param_df.sort_values('sharpe', ascending=False).to_string(index=False))

In [ ]:
# Sharpe 热图
pivot = param_df.pivot(index='slow', columns='fast', values='sharpe')

fig = px.imshow(pivot, text_auto='.3f', color_continuous_scale='RdYlGn',
                title='参数敏感性 — Sharpe 比率',
                labels={'x': 'fast window', 'y': 'slow window', 'color': 'Sharpe'})
fig.update_layout(height=350)
fig.show()

## 5. 滚动 Sharpe（策略稳定性）

In [ ]:
rolling_sharpe = analyzer.rolling_sharpe(window_bars=240 * 20).dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe.values,
                          name='20日滚动 Sharpe', line=dict(color='#9467bd')))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.add_hline(y=1, line_dash='dash', line_color='green',
               annotation_text='Sharpe=1')
fig.update_layout(title='策略滚动 Sharpe（20日）', height=380)
fig.show()

## 6. 月度收益热图

In [ ]:
eq_daily = equity.resample('1D').last().dropna()
monthly_ret = eq_daily.resample('ME').last().pct_change().dropna()

monthly_df = pd.DataFrame({
    'year': monthly_ret.index.year,
    'month': monthly_ret.index.month,
    'return': monthly_ret.values
})
pivot_m = monthly_df.pivot(index='year', columns='month', values='return') * 100
pivot_m.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                   'Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot_m.columns)]

fig = px.imshow(pivot_m, text_auto='.1f', color_continuous_scale='RdYlGn',
                color_continuous_midpoint=0,
                title='月度收益率 (%)', aspect='auto')
fig.update_layout(height=300)
fig.show()